**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Supplementary Figure: Signal Curves at Different SNR Levels

Shows a single GESFIDE signal curve (at the mean of the simulated parameter range)
under noise-free conditions and with Rician noise at SNR = 20, 50, 100, 150.

**Mean parameters used:**
- SO₂ = 50 %
- CBV = 7.625 % (midpoint of 0.25–15)
- R   = 13 µm  (midpoint of 1–25)
- T2  = 125 ms (midpoint of 50–200)

The figure is designed for the ISBI manuscript supplementary section.

## 1. Imports

In [ ]:
import numpy as np
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

print('Imports OK')

## 2. Paths — edit to match your environment

In [ ]:
DICT_PATH      = '../subsamples/subsamples_v3/QuasiRand_t2_200.mat'
PARAM_PATH     = '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat'
ECHOTIMES_PATH = '../echotimes.mat'

DICT_KEY  = 'Dico40_save'
PARAM_KEY = 'par_save'

OUTPUT_PATH = './results/figures/Fig_SNR_Visualization.png'  # where to save the figure

## 3. Load data

In [ ]:
def load_mat(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat:
            return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  Key "{key}" not found in {path}, trying "{cands[0]}"')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            if key in f:
                data = f[key][()]
            else:
                key0 = next(k for k in f if not k.startswith('#'))
                print(f'  Key "{key}" not found in {path}, trying "{key0}"')
                data = f[key0][()]
            if data.ndim >= 2:
                data = data.T
            return np.array(data, dtype=np.float32)

# --- Echo times ---
et_mat       = sio.loadmat(ECHOTIMES_PATH)
echo_times_s = et_mat['Echotimes'].flatten() / 1000.0   # convert ms → s
echo_times_ms = echo_times_s * 1000.0                   # keep ms for plotting
N_ECHOES = len(echo_times_s)
print(f'Echo times: {N_ECHOES} echoes, {echo_times_ms[0]:.2f}–{echo_times_ms[-1]:.2f} ms')

# GESFIDE segment boundaries
N_FID    = 14   # Part A: echoes  0-13
N_REPHAS = 16   # Part B: echoes 14-29  (echo 29 = spin echo)
SE_ECHO  = N_FID + N_REPHAS   # = 30  (index of first post-SE echo)
print(f'Part A: 0–{N_FID-1}, Part B: {N_FID}–{SE_ECHO-1}, Part C: {SE_ECHO}–{N_ECHOES-1}')
print(f'Spin echo at echo index {SE_ECHO-1}, t = {echo_times_ms[SE_ECHO-1]:.2f} ms')

# --- Dictionary & parameters ---
signals_nf = load_mat(DICT_PATH, DICT_KEY)   # (N, 40) noise-free
params     = load_mat(PARAM_PATH, PARAM_KEY)  # (N, 4)  [SO2, CBV, R, T2]
print(f'Dictionary: {signals_nf.shape}  |  Parameters: {params.shape}')

## 4. Select the signal closest to the mean parameter values

In [ ]:
# Parameter columns: SO2 (0–1), CBV (0.0025–0.15), R (1e-6–25e-6 m), T2 (0.05–0.20 s)
# Mean of each parameter range
PARAM_MINS = np.array([0.0,    0.0025,  1.0e-6,  0.050])
PARAM_MAXS = np.array([1.0,    0.15,   25.0e-6,  0.200])
param_means = (PARAM_MINS + PARAM_MAXS) / 2.0

print('Target (mean) parameters:')
print(f'  SO2 = {param_means[0]*100:.1f} %')
print(f'  CBV = {param_means[1]*100:.3f} %')
print(f'  R   = {param_means[2]*1e6:.1f} µm')
print(f'  T2  = {param_means[3]*1000:.1f} ms')

# Normalise parameter space to find nearest neighbour
params_norm = (params - PARAM_MINS) / (PARAM_MAXS - PARAM_MINS)
target_norm = (param_means - PARAM_MINS) / (PARAM_MAXS - PARAM_MINS)
dists = np.linalg.norm(params_norm - target_norm, axis=1)
idx   = int(np.argmin(dists))

sig_clean = signals_nf[idx].astype(np.float64)   # (40,) noise-free signal
actual_params = params[idx]

print(f'\nClosest dictionary entry (index {idx}):')
print(f'  SO2 = {actual_params[0]*100:.1f} %')
print(f'  CBV = {actual_params[1]*100:.3f} %')
print(f'  R   = {actual_params[2]*1e6:.2f} µm')
print(f'  T2  = {actual_params[3]*1000:.1f} ms')

## 5. Rician noise injection

In [ ]:
def add_rician_noise(signal, snr, n_realisations=50, rng=None):
    """
    Inject Rician noise into a 1-D signal.

    Returns:
        noisy_mean  : (40,) mean across realisations (used for single-curve display)
        noisy_all   : (n_realisations, 40) all realisations (used for shading)
    """
    if rng is None:
        rng = np.random.default_rng(42)
    S0    = signal[0]          # reference amplitude at t=0 for SNR normalisation
    sigma = S0 / snr
    nr    = rng.normal(0, sigma, (n_realisations, len(signal)))
    ni    = rng.normal(0, sigma, (n_realisations, len(signal)))
    noisy_all  = np.sqrt((signal + nr)**2 + ni**2)
    noisy_mean = noisy_all.mean(axis=0)
    return noisy_mean, noisy_all

SNR_LEVELS = [20, 50, 100, 150]
N_REALISATIONS = 200   # number of noise realisations for std shading
rng = np.random.default_rng(2024)

noisy_data = {}
for snr in SNR_LEVELS:
    mean_sig, all_sigs = add_rician_noise(sig_clean, snr,
                                          n_realisations=N_REALISATIONS, rng=rng)
    noisy_data[snr] = {'mean': mean_sig, 'all': all_sigs,
                        'std': all_sigs.std(axis=0)}
    print(f'SNR={snr:3d}  σ/S0 = {1/snr:.4f}  '
          f'avg noise (echo 0) = {all_sigs[:,0].std():.4f}')

## 6. Plot — Supplementary Figure

In [ ]:
# ── Colour scheme ──────────────────────────────────────────────────────────────
COLOR_NF  = '#1A1A1A'   # near-black  — noise-free
SNR_COLORS = {
    20:  '#D32F2F',   # red       — most noisy
    50:  '#F57C00',   # orange
    100: '#388E3C',   # green
    150: '#1565C0',   # blue      — least noisy
}
ALPHA_SHADE = 0.18    # transparency for ±1 SD shading

# ── Layout ────────────────────────────────────────────────────────────────────
# Left panel : full signal (raw amplitude, not normalised)
# Right panel: L2-normalised signal (what the network actually receives)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)

def l2norm(s):
    return s / max(np.linalg.norm(s), 1e-12)

# SE position for vertical line
t_se_ms = echo_times_ms[SE_ECHO - 1]   # spin echo time

for ax_idx, (ax, normalise) in enumerate(zip(axes, [False, True])):
    title_suffix = 'Raw amplitude' if not normalise else 'L2-normalised'

    # -- Noise-free curve --
    y_nf = l2norm(sig_clean) if normalise else sig_clean
    ax.plot(echo_times_ms, y_nf,
            color=COLOR_NF, lw=2.0, zorder=5, label='Noise-free')

    # -- Noisy curves (from lowest SNR to highest so higher-SNR stays on top) --
    for snr in SNR_LEVELS:
        c    = SNR_COLORS[snr]
        mean = noisy_data[snr]['mean']
        std  = noisy_data[snr]['std']
        if normalise:
            # Normalise each realisation independently, then take mean ± std
            all_norm = np.array([l2norm(r) for r in noisy_data[snr]['all']])
            mean = all_norm.mean(axis=0)
            std  = all_norm.std(axis=0)
        ax.fill_between(echo_times_ms, mean - std, mean + std,
                        color=c, alpha=ALPHA_SHADE, zorder=2)
        ax.plot(echo_times_ms, mean,
                color=c, lw=1.2, zorder=3, label=f'SNR = {snr}')

    # -- Spin echo marker --
    ax.axvline(t_se_ms, color='gray', lw=0.8, ls='--', zorder=1)
    ax.text(t_se_ms + 1.5, ax.get_ylim()[1] if ax_idx == 0 else 0.98,
            'SE', fontsize=7, color='gray', va='top',
            transform=ax.get_xaxis_transform())

    # -- Part labels (A / B / C) inside the plot --
    t_mid_A = (echo_times_ms[0]   + echo_times_ms[N_FID-1]) / 2
    t_mid_B = (echo_times_ms[N_FID] + echo_times_ms[SE_ECHO-1]) / 2
    t_mid_C = (echo_times_ms[SE_ECHO] + echo_times_ms[-1]) / 2
    for t_mid, lbl in [(t_mid_A, 'A'), (t_mid_B, 'B'), (t_mid_C, 'C')]:
        ax.text(t_mid, 0.02, lbl, fontsize=8, color='dimgray',
                ha='center', va='bottom', style='italic',
                transform=ax.get_xaxis_transform())

    ax.set_xlabel('Echo time (ms)')
    if not normalise:
        ax.set_ylabel('Signal amplitude (a.u.)')
    else:
        ax.set_ylabel('L2-normalised amplitude')
    ax.set_title(f'({"AB"[ax_idx]}) {title_suffix}', fontweight='bold', pad=6)
    ax.spines[['top', 'right']].set_visible(False)

# Shared legend on the right panel
handles = [mpatches.Patch(color=COLOR_NF, label='Noise-free')]
for snr in SNR_LEVELS:
    handles.append(mpatches.Patch(color=SNR_COLORS[snr], label=f'SNR = {snr}'))
axes[1].legend(handles=handles, loc='upper right', frameon=False, fontsize=8)

# Parameter annotation box
param_text = (
    f'SO₂ = {actual_params[0]*100:.0f}%  '
    f'CBV = {actual_params[1]*100:.1f}%\n'
    f'R = {actual_params[2]*1e6:.0f} µm  '
    f'T2 = {actual_params[3]*1000:.0f} ms'
)
axes[0].text(0.03, 0.97, param_text,
             transform=axes[0].transAxes,
             fontsize=7.5, va='top', ha='left',
             bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='lightgray', alpha=0.9))

fig.suptitle('GESFIDE Signal at Mean Parameter Values — Effect of SNR',
             fontsize=11, fontweight='bold', y=1.02)

plt.savefig(OUTPUT_PATH, dpi=300, bbox_inches='tight')
print(f'Figure saved → {OUTPUT_PATH}')
plt.show()

In [ ]:
# ── Colour scheme ──────────────────────────────────────────────────────────────
COLOR_NF  = '#1A1A1A'   # near-black  — noise-free
SNR_COLORS = {
    20:  '#D32F2F',   # red       — most noisy
    50:  '#F57C00',   # orange
    100: '#388E3C',   # green
    150: '#1565C0',   # blue      — least noisy
}
ALPHA_SHADE = 0.18    # transparency for ±1 SD shading

# ── Layout ────────────────────────────────────────────────────────────────────
# Left panel : full signal (raw amplitude, not normalised)
# Right panel: L2-normalised signal (what the network actually receives)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)

def l2norm(s):
    return s / max(np.linalg.norm(s), 1e-12)

# SE position for vertical line
t_se_ms = echo_times_ms[SE_ECHO - 1]   # spin echo time

for ax_idx, (ax, normalise) in enumerate(zip(axes, [False, True])):
    title_suffix = 'Raw amplitude' if not normalise else 'L2-normalised'

    # -- Noise-free curve --
    y_nf = l2norm(sig_clean) if normalise else sig_clean
    ax.plot(echo_times_ms, y_nf,
            color=COLOR_NF, lw=0.8, zorder=5, label='Noise-free')

    # -- Noisy curves (from lowest SNR to highest so higher-SNR stays on top) --
    for snr in SNR_LEVELS:
        c    = SNR_COLORS[snr]
        # mean = noisy_data[snr]['mean']
        mean = noisy_data[snr]['all'][0]   # single realisation
        std  = noisy_data[snr]['std']
        if normalise:
            # Normalise each realisation independently, then take mean ± std
            all_norm = np.array([l2norm(r) for r in noisy_data[snr]['all']])
            mean = all_norm.mean(axis=0)
            std  = all_norm.std(axis=0)
        # ax.fill_between(echo_times_ms, mean - std, mean + std,
        #                 color=c, alpha=ALPHA_SHADE, zorder=2)
        ax.plot(echo_times_ms, mean,
                color=c, lw=0.8, zorder=3, label=f'SNR = {snr}')

    # -- Spin echo marker --
    ax.axvline(t_se_ms, color='gray', lw=0.8, ls='--', zorder=1)
    ax.text(t_se_ms + 1.5, ax.get_ylim()[1] if ax_idx == 0 else 0.98,
            'SE', fontsize=7, color='gray', va='top',
            transform=ax.get_xaxis_transform())

    # -- Part labels (A / B / C) inside the plot --
    t_mid_A = (echo_times_ms[0]   + echo_times_ms[N_FID-1]) / 2
    t_mid_B = (echo_times_ms[N_FID] + echo_times_ms[SE_ECHO-1]) / 2
    t_mid_C = (echo_times_ms[SE_ECHO] + echo_times_ms[-1]) / 2
    for t_mid, lbl in [(t_mid_A, 'A'), (t_mid_B, 'B'), (t_mid_C, 'C')]:
        ax.text(t_mid, 0.02, lbl, fontsize=8, color='dimgray',
                ha='center', va='bottom', style='italic',
                transform=ax.get_xaxis_transform())

    ax.set_xlabel('Echo time (ms)')
    if not normalise:
        ax.set_ylabel('Signal amplitude (a.u.)')
    else:
        ax.set_ylabel('L2-normalised amplitude')
    ax.set_title(f'({"AB"[ax_idx]}) {title_suffix}', fontweight='bold', pad=6)
    ax.spines[['top', 'right']].set_visible(False)

# Shared legend on the right panel
handles = [mpatches.Patch(color=COLOR_NF, label='Noise-free')]
for snr in SNR_LEVELS:
    handles.append(mpatches.Patch(color=SNR_COLORS[snr], label=f'SNR = {snr}'))
axes[1].legend(handles=handles, loc='upper right', frameon=False, fontsize=8)

# Parameter annotation box
param_text = (
    f'SO2 = {actual_params[0]*100:.0f}%  '
    f'CBV = {actual_params[1]*100:.1f}%\n'
    f'R = {actual_params[2]*1e6:.0f} µm  '
    f'T2 = {actual_params[3]*1000:.0f} ms'
)
axes[0].text(0.03, 0.97, param_text,
             transform=axes[0].transAxes,
             fontsize=7.5, va='top', ha='left',
             bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='lightgray', alpha=0.9))

fig.suptitle('GESFIDE Signal at Mean Parameter Values — Effect of SNR',
             fontsize=11, fontweight='bold', y=1.02)

plt.savefig(OUTPUT_PATH, dpi=300, bbox_inches='tight')
print(f'Figure saved → {OUTPUT_PATH}')
plt.show()

## 7. (Optional) Single-panel version — raw signal only

A cleaner single-axis version if the supplementary figure only needs one panel.

In [ ]:
OUTPUT_PATH_SINGLE = './results/figures/Fig_SNR_Visualization_Single.png'

fig2, ax = plt.subplots(figsize=(5.5, 3.2), constrained_layout=True)

# Noise-free
ax.plot(echo_times_ms, sig_clean, color=COLOR_NF, lw=2.2, zorder=5, label='Noise-free')

# Noisy
for snr in SNR_LEVELS:
    c    = SNR_COLORS[snr]
    mean = noisy_data[snr]['mean']
    std  = noisy_data[snr]['std']
    ax.fill_between(echo_times_ms, mean - std, mean + std,
                    color=c, alpha=0.20, zorder=2)
    ax.plot(echo_times_ms, mean, color=c, lw=1.3, zorder=3, label=f'SNR = {snr}')

# Annotations
ax.axvline(t_se_ms, color='gray', lw=0.8, ls='--', zorder=1)
ax.text(t_se_ms + 1, 0.97, 'Spin echo', fontsize=7, color='gray',
        va='top', transform=ax.get_xaxis_transform())

for t_mid, lbl in [(t_mid_A, 'Part A'), (t_mid_B, 'Part B'), (t_mid_C, 'Part C')]:
    ax.text(t_mid, 0.03, lbl, fontsize=7.5, color='dimgray',
            ha='center', va='bottom', style='italic',
            transform=ax.get_xaxis_transform())

ax.set_xlabel('Echo time (ms)')
ax.set_ylabel('Signal amplitude (a.u.)')
ax.set_title('GESFIDE signal at mean parameters — SNR comparison', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc='upper right', frameon=False, fontsize=8)

ax.text(0.02, 0.97, param_text,
        transform=ax.transAxes, fontsize=7, va='top',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='lightgray', alpha=0.9))

plt.savefig(OUTPUT_PATH_SINGLE, dpi=300, bbox_inches='tight')
print(f'Single-panel figure saved → {OUTPUT_PATH_SINGLE}')
plt.show()